In [16]:
import os
os.listdir('../data/raw')

['ingredientsList.csv',
 'dermstore_data.json',
 'product_info_skincare.csv',
 'product_urls.csv']

In [17]:
import pandas as pd

df1 = pd.read_csv('../data/raw/ingredientsList.csv')  


In [18]:
df3 = pd.read_csv('../data/raw/product_info_skincare.csv')
print(df3.shape)
df3 = df3[df3['primary_category'] == 'Skincare'].copy()
print(df3.shape)
df3.info()
df3= df3[df3['ingredients'].notnull()].copy()

(1813, 28)
(551, 28)
<class 'pandas.DataFrame'>
Index: 551 entries, 15 to 1810
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          551 non-null    int64  
 1   product_id          551 non-null    str    
 2   product_name        551 non-null    str    
 3   brand_id            551 non-null    int64  
 4   brand_name          551 non-null    str    
 5   loves_count         551 non-null    int64  
 6   rating              541 non-null    float64
 7   reviews             541 non-null    float64
 8   size                512 non-null    str    
 9   variation_type      512 non-null    str    
 10  variation_value     499 non-null    str    
 11  variation_desc      6 non-null      str    
 12  ingredients         539 non-null    str    
 13  price_usd           551 non-null    float64
 14  value_price_usd     10 non-null     float64
 15  sale_price_usd      15 non-null     float64
 16  l

In [19]:
#this is a function for safe parsing ,mena when you convert the string into list .
def safe_parse(val):
    if pd.isnull(val):
        return []
    try :
        return ast.literal_eval(val)
    except(ValueError,SyntaxError):
        return []

In [20]:
import ast
df3['ingredients_list'] = df3['ingredients'].apply(safe_parse)
df3['highlights_list'] = df3['highlights'].apply(safe_parse)
print(df3['highlights'][19])

['Best for Dry, Combo, Normal Skin', 'Good for: Redness', 'Good for: Dryness', 'Cruelty-Free', 'Clean at Sephora', 'allure 2021 Best of Beauty Award Winner']


In [21]:
df3['highlights']

15                                                    NaN
16                                                    NaN
17                                                    NaN
18      ['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...
19      ['Best for Dry, Combo, Normal Skin', 'Good for...
                              ...                        
1806    ['Vegan', 'Refill Available', 'Good for: Pores...
1807    ['Refill Available', 'Community Favorite', 'Go...
1808    ['Vegan', 'Good for: Dullness/Uneven Texture',...
1809    ['Good for: Dullness/Uneven Texture', 'allure ...
1810    ['Vegan', 'Best for Oily Skin', 'Good for: Por...
Name: highlights, Length: 539, dtype: str

In [22]:
# this is a function for extractin the skin type for highlights
def extract_st(h):
    found = set()
    skin_keywords = ['Oily', 'Dry', 'Combo', 'Combination', 'Normal', 'Sensitive']
    for tag in h :
        if 'best for' in tag.lower():
            for kw in skin_keywords:
                if kw.lower() in tag.lower():
                    found.add(kw)
    return list(found)            

In [23]:
# this is a function for extractin the benefit 
def extract_b(h):
    found = set()
    for tag in h :
        if 'good for' in tag.lower():
           kw=tag.split(':',1)[1].strip()
           found.add(kw)
    return list(found)  

In [24]:
df3['skin_type']= df3['highlights_list'].apply(extract_st)
df3['benefit']= df3['highlights_list'].apply(extract_b)

In [25]:
df3.info()

<class 'pandas.DataFrame'>
Index: 539 entries, 15 to 1810
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          539 non-null    int64  
 1   product_id          539 non-null    str    
 2   product_name        539 non-null    str    
 3   brand_id            539 non-null    int64  
 4   brand_name          539 non-null    str    
 5   loves_count         539 non-null    int64  
 6   rating              529 non-null    float64
 7   reviews             529 non-null    float64
 8   size                507 non-null    str    
 9   variation_type      506 non-null    str    
 10  variation_value     494 non-null    str    
 11  variation_desc      6 non-null      str    
 12  ingredients         539 non-null    str    
 13  price_usd           539 non-null    float64
 14  value_price_usd     10 non-null     float64
 15  sale_price_usd      14 non-null     float64
 16  limited_edition     53

In [26]:
df1['good_for_list'] = df1['who_is_it_good_for'].apply(safe_parse)
df1['avoid_list'] = df1['who_should_avoid'].apply(safe_parse)

In [27]:
df1[['name', 'good_for_list', 'avoid_list']].head()

,name,good_for_list,avoid_list
0,Alpha-Glucan Oligosaccharide,"[ , Acne, , Blackheads, , Redness, , Pregna...","[ , Related Allergy]"
1,Aloe Vera,"[ , Dry and dehydrated skin, , Impaired skin ...","[ , Related Allergy]"
2,Allantoin,"[ , Fine Lines, , Impaired skin barrier, , R...","[ , Related Allergy]"
3,Algin,"[ , Dry and dehydrated skin, , Fine Lines, ,...","[ , Related Allergy]"
4,Algae Extract,"[ , Dry and dehydrated skin, , Fine Lines, ,...","[ , Related Allergy]"


In [28]:
def clean_list(lst):
    return [item.strip() for item in lst if item.strip() != '']

In [29]:
df1['good_for_list'] = df1['good_for_list'].apply(clean_list)
df1['avoid_list'] = df1['avoid_list'].apply(clean_list)

df1[['name', 'good_for_list', 'avoid_list']].head()

,name,good_for_list,avoid_list
0,Alpha-Glucan Oligosaccharide,"[Acne, Blackheads, Redness, Pregnancy]",[Related Allergy]
1,Aloe Vera,"[Dry and dehydrated skin, Impaired skin barrie...",[Related Allergy]
2,Allantoin,"[Fine Lines, Impaired skin barrier, Redness, P...",[Related Allergy]
3,Algin,"[Dry and dehydrated skin, Fine Lines, Pregnanc...",[Related Allergy]
4,Algae Extract,"[Dry and dehydrated skin, Fine Lines, Pregnanc...",[Related Allergy]


In [30]:
df1.head(10)

,name,scientific_name,short_description,what_is_it,what_does_it_do,who_is_it_good_for,who_should_avoid,url,good_for_list,avoid_list
0,Alpha-Glucan Oligosaccharide,NaN,Alpha-glucan oligosaccharide is in a class of ...,Prebiotics are a type of non-digestible dietar...,Prebiotics offer benefits such as:\r\n\r\n- He...,"[' ', 'Acne', ' ', 'Blackheads', ' ', 'Redness...","[' ', 'Related Allergy']",https://renude.co/ingredients/alpha-glucan-oli...,"[Acne, Blackheads, Redness, Pregnancy]",[Related Allergy]
1,Aloe Vera,NaN,"Aloe vera, also appear on ingredients lists as...",Aloe vera is a skincare ingredient derived fro...,Aloe vera offers multiple benefits for the ski...,"[' ', 'Dry and dehydrated skin', ' ', 'Impaire...","[' ', 'Related Allergy']",https://renude.co/ingredients/aloe-vera,"[Dry and dehydrated skin, Impaired skin barrie...",[Related Allergy]
2,Allantoin,NaN,"Allantoin occurs naturally in the body, but ca...",Allantoin is a skincare ingredient derived fro...,"Allantoin is a calming, anti-inflammatory, and...","[' ', 'Fine Lines', ' ', 'Impaired skin barrie...","[' ', 'Related Allergy']",https://renude.co/ingredients/allantoin,"[Fine Lines, Impaired skin barrier, Redness, P...",[Related Allergy]
3,Algin,NaN,"Algin, also known as sodium alginate, is a lar...",An extract from brown seaweed used as hydratin...,"In skincare products, it is used for its excel...","[' ', 'Dry and dehydrated skin', ' ', 'Fine Li...","[' ', 'Related Allergy']",https://renude.co/ingredients/algin,"[Dry and dehydrated skin, Fine Lines, Pregnanc...",[Related Allergy]
4,Algae Extract,NaN,"It is essentially an underwater plant, designe...",An incredibly interesting natural ingredient s...,Algae extracts are multifunctional ingredients...,"[' ', 'Dry and dehydrated skin', ' ', 'Fine Li...","[' ', 'Related Allergy']",https://renude.co/ingredients/algae-extract,"[Dry and dehydrated skin, Fine Lines, Pregnanc...",[Related Allergy]
5,Algae Exopolysaccharides,NaN,Exopolysaccharides (EPS) are phytochemicals pr...,EPSs are macromolecular polysaccharidic compou...,"In skincare, they are are used to protect from...","[' ', 'Dry and dehydrated skin', ' ', 'Fine Li...","[' ', 'Related Allergy']",https://renude.co/ingredients/algae-exopolysac...,"[Dry and dehydrated skin, Fine Lines, Pregnanc...",[Related Allergy]
6,Alaria Esculenta Extract,NaN,"A type of edible brown algae, harvested from w...",A type of edible brown algae composed of amino...,"Provides soothing, hydrating, firming and anti...","[' ', 'Dry and dehydrated skin', ' ', 'Fine Li...","[' ', 'Related Allergy']",https://renude.co/ingredients/alaria-esculenta...,"[Dry and dehydrated skin, Fine Lines, Pregnanc...",[Related Allergy]
7,Alanine,NaN,Amino acids like Alanine are the small buildin...,"Alanine is present in the ""natural moisturisin...",It can help to support water transport between...,"[' ', 'Dry and dehydrated skin', ' ', 'Pregnan...","[' ', 'Related Allergy']",https://renude.co/ingredients/alanine,"[Dry and dehydrated skin, Pregnancy, Texture]",[Related Allergy]
8,Ahnfeltia Concinna Extract,NaN,"Ahnfeltia concinna extract, or red marine alga...",A natural extract from red marine algae with m...,Provides soothing and antioxidant products to ...,"[' ', 'Dry and dehydrated skin', ' ', 'Fine Li...","[' ', 'Related Allergy']",https://renude.co/ingredients/ahnfeltia-concin...,"[Dry and dehydrated skin, Fine Lines, Pregnanc...",[Related Allergy]
9,Adenosine,NaN,Adenosine is a well-researched anti-ageing and...,Adenosine is a skincare ingredient derived fro...,Provides soothing and skin-restoring propertie...,"[' ', 'Acne', ' ', 'Dry and dehydrated skin', ...","[' ', 'Related Allergy']",https://renude.co/ingredients/adenosine,"[Acne, Dry and dehydrated skin, Elasticity, Fi...",[Related Allergy]


In [31]:
df1 = df1[['name', 'good_for_list', 'avoid_list']].copy()
def fix_typos(lst):
    return ['Gluten Allergy' if item == 'Gluten Allery' else item for item in lst]

df1['avoid_list'] = df1['avoid_list'].apply(fix_typos)

In [32]:
df3.to_csv('../data/processed/skincare_clean.csv', index=False)
df1.to_csv('../data/processed/ingredients_reference_clean.csv', index=False)

In [33]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   name           247 non-null    str   
 1   good_for_list  248 non-null    object
 2   avoid_list     248 non-null    object
dtypes: object(2), str(1)
memory usage: 5.9+ KB


In [34]:
df3.info()

<class 'pandas.DataFrame'>
Index: 539 entries, 15 to 1810
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          539 non-null    int64  
 1   product_id          539 non-null    str    
 2   product_name        539 non-null    str    
 3   brand_id            539 non-null    int64  
 4   brand_name          539 non-null    str    
 5   loves_count         539 non-null    int64  
 6   rating              529 non-null    float64
 7   reviews             529 non-null    float64
 8   size                507 non-null    str    
 9   variation_type      506 non-null    str    
 10  variation_value     494 non-null    str    
 11  variation_desc      6 non-null      str    
 12  ingredients         539 non-null    str    
 13  price_usd           539 non-null    float64
 14  value_price_usd     10 non-null     float64
 15  sale_price_usd      14 non-null     float64
 16  limited_edition     53

In [35]:
type(df3['benefit'].iloc[0])

list

In [36]:
print('Task 2/3 usable rows (has skin_type):', (df3['skin_type'].apply(len) > 0).sum())
print('Task 2/3 usable rows (has benefit):', (df3['benefit'].apply(len) > 0).sum())
print('Task 1 usable rows (has ingredients):', (df3['ingredients_list'].apply(len) > 0).sum())

Task 2/3 usable rows (has skin_type): 203
Task 2/3 usable rows (has benefit): 380
Task 1 usable rows (has ingredients): 539


In [37]:
df1.head()

,name,good_for_list,avoid_list
0,Alpha-Glucan Oligosaccharide,"[Acne, Blackheads, Redness, Pregnancy]",[Related Allergy]
1,Aloe Vera,"[Dry and dehydrated skin, Impaired skin barrie...",[Related Allergy]
2,Allantoin,"[Fine Lines, Impaired skin barrier, Redness, P...",[Related Allergy]
3,Algin,"[Dry and dehydrated skin, Fine Lines, Pregnanc...",[Related Allergy]
4,Algae Extract,"[Dry and dehydrated skin, Fine Lines, Pregnanc...",[Related Allergy]


In [38]:
df3.head()

,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,ingredients_list,highlights_list,skin_type,benefit
15,95,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,6018,Algenist,9314,4.0569,334.0,5 oz/ 150 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[C12-15 Alkyl Benzoate, Ethylhexyl Palmitate, ...",[],[],[]
16,96,P296413,Gentle Rejuvenating Cleanser,6018,Algenist,7681,4.2689,264.0,4 oz/ 120 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Sodium Cocoyl Isethionate, Glyceryl St...",[],[],[]
17,98,P379907,Advanced Anti-Aging Repairing Oil,6018,Algenist,10676,4.4531,245.0,1 oz/ 30 mL,Size,...,Skincare,Moisturizers,Face Oils,0,NaN,NaN,"[Chlorella Protothecoides Oil, Cetearyl Ethylh...",[],[],[]
18,107,P442859,ALIVE Prebiotic Balancing Mask,6018,Algenist,14367,4.3729,118.0,1.7 oz/ 50 mL,Size,...,Skincare,Masks,Face Masks,0,NaN,NaN,"[Glycerin, Water (Aqua, Eau), Sodium Cocoyl Gl...","[Vegan, Hypoallergenic, Good for: Acne/Blemish...",[Combination],[Acne/Blemishes]
19,125,P442546,Balancing Cleanser,6283,Alpha-H,3612,4.5455,77.0,6.25 oz/ 185 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Ethylhexyl Palmitate, Aloe Barbadensis...","[Best for Dry, Combo, Normal Skin, Good for: R...","[Combo, Normal, Dry]","[Redness, Dryness]"


In [39]:
df3.to_csv('../data/processed/skincare_clean.csv', index=False)

In [40]:
print(df3.columns.tolist())

['Unnamed: 0', 'product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price', 'ingredients_list', 'highlights_list', 'skin_type', 'benefit']


In [41]:
print(df3.columns.tolist())

['Unnamed: 0', 'product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price', 'ingredients_list', 'highlights_list', 'skin_type', 'benefit']


In [42]:
import pandas as pd
import ast

def safe_parse(val):
    if pd.isnull(val):
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

df3 = pd.read_csv('../data/processed/skincare_clean.csv')
df3['ingredients_list'] = df3['ingredients_list'].apply(safe_parse)
df3['skin_type'] = df3['skin_type'].apply(safe_parse)
df3['benefit'] = df3['benefit'].apply(safe_parse)

print(df3.shape)
print(df3.columns.tolist())

(539, 32)
['Unnamed: 0', 'product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price', 'ingredients_list', 'highlights_list', 'skin_type', 'benefit']


In [43]:
df1 = pd.read_csv('../data/raw/ingredientsList.csv')
print(df1.shape)
print(df1.columns.tolist())


(248, 8)
['name', 'scientific_name', 'short_description', 'what_is_it', 'what_does_it_do', 'who_is_it_good_for', 'who_should_avoid', 'url']


In [44]:
df1['good_for_list'] = df1['who_is_it_good_for'].apply(safe_parse)
df1['avoid_list'] = df1['who_should_avoid'].apply(safe_parse)

print(type(df1['good_for_list'].iloc[0]))
print(df1[['name', 'good_for_list', 'avoid_list']].head(3))

<class 'list'>
                           name  \
0  Alpha-Glucan Oligosaccharide   
1                     Aloe Vera   
2                     Allantoin   

                                       good_for_list            avoid_list  
0  [ , Acne,  , Blackheads,  , Redness,  , Pregna...  [ , Related Allergy]  
1  [ , Dry and dehydrated skin,  , Impaired skin ...  [ , Related Allergy]  
2  [ , Fine Lines,  , Impaired skin barrier,  , R...  [ , Related Allergy]  


In [45]:
def clean_list(lst):
    return [item.strip() for item in lst if item.strip() != '']

df1['good_for_list'] = df1['good_for_list'].apply(clean_list)
df1['avoid_list'] = df1['avoid_list'].apply(clean_list)

df1[['name', 'good_for_list', 'avoid_list']].head(3)


,name,good_for_list,avoid_list
0,Alpha-Glucan Oligosaccharide,"[Acne, Blackheads, Redness, Pregnancy]",[Related Allergy]
1,Aloe Vera,"[Dry and dehydrated skin, Impaired skin barrie...",[Related Allergy]
2,Allantoin,"[Fine Lines, Impaired skin barrier, Redness, P...",[Related Allergy]


In [46]:
def fix_typos(lst):
    return ['Gluten Allergy' if item == 'Gluten Allery' else item for item in lst]

df1['avoid_list'] = df1['avoid_list'].apply(fix_typos)

In [47]:
df1_clean = df1[['name', 'good_for_list', 'avoid_list']].copy()

df1_clean.to_csv('../data/processed/ingredients_reference_clean.csv', index=False)

print(df1_clean.shape)
df1_clean.head(3)

(248, 3)


,name,good_for_list,avoid_list
0,Alpha-Glucan Oligosaccharide,"[Acne, Blackheads, Redness, Pregnancy]",[Related Allergy]
1,Aloe Vera,"[Dry and dehydrated skin, Impaired skin barrie...",[Related Allergy]
2,Allantoin,"[Fine Lines, Impaired skin barrier, Redness, P...",[Related Allergy]


In [48]:
print(len(df3['ingredients_list'].iloc[0]))

1


In [49]:
def split_ingredients(lst):
    text = lst[0] if len(lst) > 0 else ''
    return [i.strip() for i in text.split(',') if i.strip()]

df3['ingredients_list'] = df3['ingredients_list'].apply(split_ingredients)

print(len(df3['ingredients_list'].iloc[0]))
df3['ingredients_list'].iloc[0]

20


['C12-15 Alkyl Benzoate',
 'Ethylhexyl Palmitate',
 'Caprylic/Capric Triglyceride',
 'Glycerin',
 'Water (Aqua)',
 'Sucrose Laurate',
 'Olea Europaea (Olive) Fruit Oil',
 'Persea Gratissima (Avocado) Oil',
 'Chlorella Protothecoides Oil',
 'Algae Exopolysaccharides',
 'Prunus Amygdalus Dulcis (Sweet Almond) Oil',
 'Vaccinium Myrtillus Seed Oil',
 'Retinyl Palmitate',
 'Tocopheryl Acetate',
 'Caprylyl Glycol',
 'Chlorphenesin',
 'Phenoxyethanol',
 'Fragrance (Parfum)',
 'Benzyl Benzoate',
 'Chromium Oxide Greens (CI 77288).']

In [50]:
df3.to_csv('../data/processed/skincare_clean.csv', index=False)

In [51]:
print('df3 shape:', df3.shape)
print('df1 shape:', df1.shape)
print()

# Confirm ingredients are properly split (not one blob)
print('Sample ingredient count for product 0:', len(df3['ingredients_list'].iloc[0]))
print()

# Real match check now that ingredients are actually split
df3_ingredients = set()
for lst in df3['ingredients_list']:
    df3_ingredients.update([i.lower().strip() for i in lst])

df1_names = set(df1['name'].str.lower().str.strip())

matched = df3_ingredients & df1_names
print('Unique ingredients in df3:', len(df3_ingredients))
print('Unique ingredient names in df1:', len(df1_names))
print('Exact matches:', len(matched))

df3 shape: (539, 32)
df1 shape: (248, 10)

Sample ingredient count for product 0: 20

Unique ingredients in df3: 3042
Unique ingredient names in df1: 248
Exact matches: 93


In [52]:
df1_clean.shape

(248, 3)

In [53]:
%whos

Variable                  Type                     Data/Info
------------------------------------------------------------
KNOWN_COMMA_INGREDIENTS   list                     n=4
LogisticRegression        type                     <class 'sklearn.linear_mo<...>stic.LogisticRegression'>
MultiOutputClassifier     ABCMeta                  <class 'sklearn.multioutp<...>t.MultiOutputClassifier'>
X_empty                   ndarray                  1x405: 405 elems, type `int64`, 3240 bytes
ast                       module                   <module 'ast' from '/opt/<...>4/lib/python3.14/ast.py'>
base_model                LogisticRegression       LogisticRegression(class_<...>er=1000, random_state=42)
clean_list                function                 <function clean_list at 0x11685f8a0>
df1                       DataFrame                Shape: (248, 10)
df1_clean                 DataFrame                Shape: (248, 3)
df1_names                 set                      {'cinnamon bark', 'sweet <.

In [54]:
df3.head()

,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,ingredients_list,highlights_list,skin_type,benefit
0,95,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,6018,Algenist,9314,4.0569,334.0,5 oz/ 150 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[C12-15 Alkyl Benzoate, Ethylhexyl Palmitate, ...",[],[],[]
1,96,P296413,Gentle Rejuvenating Cleanser,6018,Algenist,7681,4.2689,264.0,4 oz/ 120 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Sodium Cocoyl Isethionate, Glyceryl St...",[],[],[]
2,98,P379907,Advanced Anti-Aging Repairing Oil,6018,Algenist,10676,4.4531,245.0,1 oz/ 30 mL,Size,...,Skincare,Moisturizers,Face Oils,0,NaN,NaN,"[Chlorella Protothecoides Oil, Cetearyl Ethylh...",[],[],[]
3,107,P442859,ALIVE Prebiotic Balancing Mask,6018,Algenist,14367,4.3729,118.0,1.7 oz/ 50 mL,Size,...,Skincare,Masks,Face Masks,0,NaN,NaN,"[Glycerin, Water (Aqua, Eau), Sodium Cocoyl Gl...","['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...",[Combination],[Acne/Blemishes]
4,125,P442546,Balancing Cleanser,6283,Alpha-H,3612,4.5455,77.0,6.25 oz/ 185 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Ethylhexyl Palmitate, Aloe Barbadensis...","['Best for Dry, Combo, Normal Skin', 'Good for...","[Combo, Normal, Dry]","[Redness, Dryness]"


In [55]:
%whos


Variable                  Type                     Data/Info
------------------------------------------------------------
KNOWN_COMMA_INGREDIENTS   list                     n=4
LogisticRegression        type                     <class 'sklearn.linear_mo<...>stic.LogisticRegression'>
MultiOutputClassifier     ABCMeta                  <class 'sklearn.multioutp<...>t.MultiOutputClassifier'>
X_empty                   ndarray                  1x405: 405 elems, type `int64`, 3240 bytes
ast                       module                   <module 'ast' from '/opt/<...>4/lib/python3.14/ast.py'>
base_model                LogisticRegression       LogisticRegression(class_<...>er=1000, random_state=42)
clean_list                function                 <function clean_list at 0x11685f8a0>
df1                       DataFrame                Shape: (248, 10)
df1_clean                 DataFrame                Shape: (248, 3)
df1_names                 set                      {'cinnamon bark', 'sweet <.

In [56]:
df1 = pd.read_csv('../data/processed/ingredients_reference_clean.csv') 

In [57]:
df3 = pd.read_csv('../data/processed/skincare_clean.csv') 

In [58]:
df1

,name,good_for_list,avoid_list
0,Alpha-Glucan Oligosaccharide,"['Acne', 'Blackheads', 'Redness', 'Pregnancy']",['Related Allergy']
1,Aloe Vera,"['Dry and dehydrated skin', 'Impaired skin bar...",['Related Allergy']
2,Allantoin,"['Fine Lines', 'Impaired skin barrier', 'Redne...",['Related Allergy']
3,Algin,"['Dry and dehydrated skin', 'Fine Lines', 'Pre...",['Related Allergy']
4,Algae Extract,"['Dry and dehydrated skin', 'Fine Lines', 'Pre...",['Related Allergy']
...,...,...,...
243,Zinc Sulfate,['Anyone'],['Related Allergy']
244,Zinc PCA,"['Acne', 'Blackheads', 'Enlarged Pores', 'Fine...",['Related Allergy']
245,Neoglucosamine,"['Blackheads', 'Elasticity', 'Enlarged Pores',...","['Related Allergy', 'Impaired skin barrier']"
246,Taurine,"['Dry and dehydrated skin', 'Radiance', 'Pregn...",['Related Allergy']


In [59]:
df3

,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,ingredients_list,highlights_list,skin_type,benefit
0,95,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,6018,Algenist,9314,4.0569,334.0,5 oz/ 150 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"['C12-15 Alkyl Benzoate', 'Ethylhexyl Palmitat...",[],[],[]
1,96,P296413,Gentle Rejuvenating Cleanser,6018,Algenist,7681,4.2689,264.0,4 oz/ 120 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"['Water', 'Sodium Cocoyl Isethionate', 'Glycer...",[],[],[]
2,98,P379907,Advanced Anti-Aging Repairing Oil,6018,Algenist,10676,4.4531,245.0,1 oz/ 30 mL,Size,...,Skincare,Moisturizers,Face Oils,0,NaN,NaN,"['Chlorella Protothecoides Oil', 'Cetearyl Eth...",[],[],[]
3,107,P442859,ALIVE Prebiotic Balancing Mask,6018,Algenist,14367,4.3729,118.0,1.7 oz/ 50 mL,Size,...,Skincare,Masks,Face Masks,0,NaN,NaN,"['Glycerin', 'Water (Aqua', 'Eau)', 'Sodium Co...","['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...",['Combination'],['Acne/Blemishes']
4,125,P442546,Balancing Cleanser,6283,Alpha-H,3612,4.5455,77.0,6.25 oz/ 185 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"['Water', 'Ethylhexyl Palmitate', 'Aloe Barbad...","['Best for Dry, Combo, Normal Skin', 'Good for...","['Combo', 'Normal', 'Dry']","['Redness', 'Dryness']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,8402,P411387,Superfood Antioxidant Cleanser,6169,Youth To The People,404142,4.2112,5851.0,8 oz/ 237 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,2,68.0,14.0,"['Water/Aqua/Eau', 'Cocamidopropyl Hydroxysult...","['Vegan', 'Refill Available', 'Good for: Pores...","['Normal', 'Oily', 'Combo']",['Pores']
535,8403,P441644,Mini Superfood Antioxidant Cleanser,6169,Youth To The People,121678,4.2121,5841.0,2 oz/ 59 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,"['Water/Aqua/Eau', 'Cocamidopropyl Hydroxysult...","['Refill Available', 'Community Favorite', 'Go...","['Normal', 'Oily', 'Combo']",['Pores']
536,8404,P440307,Superberry Hydrate + Glow Dream Night Mask wit...,6169,Youth To The People,248887,4.3237,2354.0,2 oz/ 59 mL,Size,...,Skincare,Masks,Face Masks,1,18.0,18.0,"['Water/Aqua/Eau', 'Glycerin', 'Helianthus Ann...","['Vegan', 'Good for: Dullness/Uneven Texture',...",[],['Dullness/Uneven Texture']
537,8405,P461555,Mini Superberry Hydrate + Glow Dream Mask,6169,Youth To The People,79524,4.3237,2354.0,0.5 oz/ 15 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,"['Water/Aqua/Eau', 'Glycerin', 'Helianthus Ann...","['Good for: Dullness/Uneven Texture', 'allure ...","['Combo', 'Normal', 'Dry']",['Dullness/Uneven Texture']


In [60]:
def normalize_skin_type(lst):
    mapping = {
        'Combo': 'Combination',
        'Combination': 'Combination',
        'Normal': 'Normal',
        'Dry': 'Dry',
        'Oily': 'Oily',
        'Sensitive': 'Sensitive'
    }
    return list(set(mapping.get(item, item) for item in lst))

df3['skin_type'] = df3['skin_type'].apply(normalize_skin_type)

In [61]:
from collections import Counter
all_skin_types = []
for lst in df3['skin_type']:
    all_skin_types.extend(lst)
print(Counter(all_skin_types))

Counter({']': 539, '[': 539, "'": 203, 'y': 196, 'r': 185, 'l': 180, 'o': 172, 'm': 172, 'a': 172, 'b': 170, 'C': 170, 'N': 167, ' ': 165, ',': 165, 'D': 108, 'i': 100, 'O': 95, 'n': 5, 't': 5})


In [62]:
import pandas as pd
import ast

def safe_parse(val):
    if pd.isnull(val):
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

df3 = pd.read_csv('../data/processed/skincare_clean.csv')
df3['ingredients_list'] = df3['ingredients_list'].apply(safe_parse)
df3['skin_type'] = df3['skin_type'].apply(safe_parse)
df3['benefit'] = df3['benefit'].apply(safe_parse)

# Confirm it's a real list now
print(type(df3['skin_type'].iloc[0]))
print(df3['skin_type'].iloc[3])  # should show something like ['Combination']

<class 'list'>
['Combination']


In [63]:
def normalize_skin_type(lst):
    mapping = {
        'Combo': 'Combination',
        'Combination': 'Combination',
        'Normal': 'Normal',
        'Dry': 'Dry',
        'Oily': 'Oily',
        'Sensitive': 'Sensitive'
    }
    return list(set(mapping.get(item, item) for item in lst))

df3['skin_type'] = df3['skin_type'].apply(normalize_skin_type)

In [64]:
from collections import Counter
all_skin_types = []
for lst in df3['skin_type']:
    all_skin_types.extend(lst)
print(Counter(all_skin_types))

Counter({'Combination': 170, 'Normal': 167, 'Dry': 108, 'Oily': 95})


In [65]:
df3.to_csv('../data/processed/skincare_clean.csv', index=False)

In [66]:
all_benefits = []
for lst in df3['benefit']:
    all_benefits.extend(lst)
print(Counter(all_benefits))

Counter({'Dryness': 163, 'Dullness/Uneven Texture': 155, 'Pores': 114, 'Acne/Blemishes': 61, 'Anti-Aging': 60, 'Loss of firmness': 50, 'Dark Circles': 42, 'Redness': 40, 'Dark spots': 20})


In [67]:
import pandas as pd
df3= pd.read_csv('../data/processed/skincare_clean.csv')

In [68]:
df3

,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,ingredients_list,highlights_list,skin_type,benefit
0,95,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,6018,Algenist,9314,4.0569,334.0,5 oz/ 150 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"['C12-15 Alkyl Benzoate', 'Ethylhexyl Palmitat...",[],[],[]
1,96,P296413,Gentle Rejuvenating Cleanser,6018,Algenist,7681,4.2689,264.0,4 oz/ 120 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"['Water', 'Sodium Cocoyl Isethionate', 'Glycer...",[],[],[]
2,98,P379907,Advanced Anti-Aging Repairing Oil,6018,Algenist,10676,4.4531,245.0,1 oz/ 30 mL,Size,...,Skincare,Moisturizers,Face Oils,0,NaN,NaN,"['Chlorella Protothecoides Oil', 'Cetearyl Eth...",[],[],[]
3,107,P442859,ALIVE Prebiotic Balancing Mask,6018,Algenist,14367,4.3729,118.0,1.7 oz/ 50 mL,Size,...,Skincare,Masks,Face Masks,0,NaN,NaN,"['Glycerin', 'Water (Aqua', 'Eau)', 'Sodium Co...","['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...",['Combination'],['Acne/Blemishes']
4,125,P442546,Balancing Cleanser,6283,Alpha-H,3612,4.5455,77.0,6.25 oz/ 185 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"['Water', 'Ethylhexyl Palmitate', 'Aloe Barbad...","['Best for Dry, Combo, Normal Skin', 'Good for...","['Normal', 'Combination', 'Dry']","['Redness', 'Dryness']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,8402,P411387,Superfood Antioxidant Cleanser,6169,Youth To The People,404142,4.2112,5851.0,8 oz/ 237 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,2,68.0,14.0,"['Water/Aqua/Eau', 'Cocamidopropyl Hydroxysult...","['Vegan', 'Refill Available', 'Good for: Pores...","['Combination', 'Normal', 'Oily']",['Pores']
535,8403,P441644,Mini Superfood Antioxidant Cleanser,6169,Youth To The People,121678,4.2121,5841.0,2 oz/ 59 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,"['Water/Aqua/Eau', 'Cocamidopropyl Hydroxysult...","['Refill Available', 'Community Favorite', 'Go...","['Combination', 'Normal', 'Oily']",['Pores']
536,8404,P440307,Superberry Hydrate + Glow Dream Night Mask wit...,6169,Youth To The People,248887,4.3237,2354.0,2 oz/ 59 mL,Size,...,Skincare,Masks,Face Masks,1,18.0,18.0,"['Water/Aqua/Eau', 'Glycerin', 'Helianthus Ann...","['Vegan', 'Good for: Dullness/Uneven Texture',...",[],['Dullness/Uneven Texture']
537,8405,P461555,Mini Superberry Hydrate + Glow Dream Mask,6169,Youth To The People,79524,4.3237,2354.0,0.5 oz/ 15 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,"['Water/Aqua/Eau', 'Glycerin', 'Helianthus Ann...","['Good for: Dullness/Uneven Texture', 'allure ...","['Normal', 'Combination', 'Dry']",['Dullness/Uneven Texture']


In [69]:
from collections import Counter
ingredient_counts = Counter()
for lst in df3['ingredients_list']:
    unique_in_product = set(i.lower().strip() for i in lst)
    ingredient_counts.update(unique_in_product)

print('Total unique ingredients:', len(ingredient_counts))

# Keep only ingredients appearing in at least 5 products
MIN_PRODUCTS = 5
frequent_ingredients = {ing for ing, count in ingredient_counts.items() if count >= MIN_PRODUCTS}
print('Ingredients appearing in 5+ products:', len(frequent_ingredients))

Total unique ingredients: 63
Ingredients appearing in 5+ products: 55


In [70]:
print(df3['ingredients_list'].iloc[0])
print(type(df3['ingredients_list'].iloc[0]))
print(len(df3['ingredients_list'].iloc[0]))

['C12-15 Alkyl Benzoate', 'Ethylhexyl Palmitate', 'Caprylic/Capric Triglyceride', 'Glycerin', 'Water (Aqua)', 'Sucrose Laurate', 'Olea Europaea (Olive) Fruit Oil', 'Persea Gratissima (Avocado) Oil', 'Chlorella Protothecoides Oil', 'Algae Exopolysaccharides', 'Prunus Amygdalus Dulcis (Sweet Almond) Oil', 'Vaccinium Myrtillus Seed Oil', 'Retinyl Palmitate', 'Tocopheryl Acetate', 'Caprylyl Glycol', 'Chlorphenesin', 'Phenoxyethanol', 'Fragrance (Parfum)', 'Benzyl Benzoate', 'Chromium Oxide Greens (CI 77288).']
<class 'str'>
511


In [71]:
print(df3['ingredients_list'].head(3).to_list())

["['C12-15 Alkyl Benzoate', 'Ethylhexyl Palmitate', 'Caprylic/Capric Triglyceride', 'Glycerin', 'Water (Aqua)', 'Sucrose Laurate', 'Olea Europaea (Olive) Fruit Oil', 'Persea Gratissima (Avocado) Oil', 'Chlorella Protothecoides Oil', 'Algae Exopolysaccharides', 'Prunus Amygdalus Dulcis (Sweet Almond) Oil', 'Vaccinium Myrtillus Seed Oil', 'Retinyl Palmitate', 'Tocopheryl Acetate', 'Caprylyl Glycol', 'Chlorphenesin', 'Phenoxyethanol', 'Fragrance (Parfum)', 'Benzyl Benzoate', 'Chromium Oxide Greens (CI 77288).']", "['Water', 'Sodium Cocoyl Isethionate', 'Glyceryl Stearate SE', 'Stearic Acid', 'Cocamidopropyl Betaine', 'Cetyl Alcohol', 'Dimethicone PEG-8 Meadowfoamate', 'Hamamelis Virginiana Water (Hamamelis Virginiana (Witch Hazel) Water)', 'Algae Exopolysaccharides', 'Tocopheryl Acetate', 'Sodium Lauroyl Oat Amino Acids', 'Camellia Sinensis Leaf Extract', 'Pyrus Malus Fruit Extract (Pyrus Malus (Apple) Fruit Extract)', 'Citrus Medica Limonum Fruit Extract (Citrus Medica Limonum (Lemon) Fr

In [72]:
import pandas as pd
import ast

def safe_parse(val):
    if pd.isnull(val):
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

df3 = pd.read_csv('../data/processed/skincare_clean.csv')
df3['ingredients_list'] = df3['ingredients_list'].apply(safe_parse)
df3['skin_type'] = df3['skin_type'].apply(safe_parse)
df3['benefit'] = df3['benefit'].apply(safe_parse)

print(df3.shape)
print(df3.columns.tolist())

(539, 32)
['Unnamed: 0', 'product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price', 'ingredients_list', 'highlights_list', 'skin_type', 'benefit']


In [73]:
def split_ingredients(lst):
    text = lst[0] if len(lst) > 0 else ''
    return [i.strip() for i in text.split(',') if i.strip()]

df3['ingredients_list'] = df3['ingredients_list'].apply(split_ingredients)

print(len(df3['ingredients_list'].iloc[0]))
df3['ingredients_list'].iloc[0]

1


['C12-15 Alkyl Benzoate']

In [74]:
df3

,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,ingredients_list,highlights_list,skin_type,benefit
0,95,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,6018,Algenist,9314,4.0569,334.0,5 oz/ 150 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,[C12-15 Alkyl Benzoate],[],[],[]
1,96,P296413,Gentle Rejuvenating Cleanser,6018,Algenist,7681,4.2689,264.0,4 oz/ 120 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,[Water],[],[],[]
2,98,P379907,Advanced Anti-Aging Repairing Oil,6018,Algenist,10676,4.4531,245.0,1 oz/ 30 mL,Size,...,Skincare,Moisturizers,Face Oils,0,NaN,NaN,[Chlorella Protothecoides Oil],[],[],[]
3,107,P442859,ALIVE Prebiotic Balancing Mask,6018,Algenist,14367,4.3729,118.0,1.7 oz/ 50 mL,Size,...,Skincare,Masks,Face Masks,0,NaN,NaN,[Glycerin],"['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...",[Combination],[Acne/Blemishes]
4,125,P442546,Balancing Cleanser,6283,Alpha-H,3612,4.5455,77.0,6.25 oz/ 185 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,[Water],"['Best for Dry, Combo, Normal Skin', 'Good for...","[Normal, Combination, Dry]","[Redness, Dryness]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,8402,P411387,Superfood Antioxidant Cleanser,6169,Youth To The People,404142,4.2112,5851.0,8 oz/ 237 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,2,68.0,14.0,[Water/Aqua/Eau],"['Vegan', 'Refill Available', 'Good for: Pores...","[Combination, Normal, Oily]",[Pores]
535,8403,P441644,Mini Superfood Antioxidant Cleanser,6169,Youth To The People,121678,4.2121,5841.0,2 oz/ 59 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,[Water/Aqua/Eau],"['Refill Available', 'Community Favorite', 'Go...","[Combination, Normal, Oily]",[Pores]
536,8404,P440307,Superberry Hydrate + Glow Dream Night Mask wit...,6169,Youth To The People,248887,4.3237,2354.0,2 oz/ 59 mL,Size,...,Skincare,Masks,Face Masks,1,18.0,18.0,[Water/Aqua/Eau],"['Vegan', 'Good for: Dullness/Uneven Texture',...",[],[Dullness/Uneven Texture]
537,8405,P461555,Mini Superberry Hydrate + Glow Dream Mask,6169,Youth To The People,79524,4.3237,2354.0,0.5 oz/ 15 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,[Water/Aqua/Eau],"['Good for: Dullness/Uneven Texture', 'allure ...","[Normal, Combination, Dry]",[Dullness/Uneven Texture]


In [75]:
df3 = pd.read_csv('../data/processed/skincare_clean.csv')

In [76]:
type (df3['ingredients_list'][0])

str

In [77]:
import ast

df3['ingredients_list'] = df3['ingredients_list'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [78]:
type(df3['ingredients_list'].iloc[0])

list

In [79]:
df3

,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,ingredients_list,highlights_list,skin_type,benefit
0,95,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,6018,Algenist,9314,4.0569,334.0,5 oz/ 150 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[C12-15 Alkyl Benzoate, Ethylhexyl Palmitate, ...",[],[],[]
1,96,P296413,Gentle Rejuvenating Cleanser,6018,Algenist,7681,4.2689,264.0,4 oz/ 120 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Sodium Cocoyl Isethionate, Glyceryl St...",[],[],[]
2,98,P379907,Advanced Anti-Aging Repairing Oil,6018,Algenist,10676,4.4531,245.0,1 oz/ 30 mL,Size,...,Skincare,Moisturizers,Face Oils,0,NaN,NaN,"[Chlorella Protothecoides Oil, Cetearyl Ethylh...",[],[],[]
3,107,P442859,ALIVE Prebiotic Balancing Mask,6018,Algenist,14367,4.3729,118.0,1.7 oz/ 50 mL,Size,...,Skincare,Masks,Face Masks,0,NaN,NaN,"[Glycerin, Water (Aqua, Eau), Sodium Cocoyl Gl...","['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...",['Combination'],['Acne/Blemishes']
4,125,P442546,Balancing Cleanser,6283,Alpha-H,3612,4.5455,77.0,6.25 oz/ 185 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Ethylhexyl Palmitate, Aloe Barbadensis...","['Best for Dry, Combo, Normal Skin', 'Good for...","['Normal', 'Combination', 'Dry']","['Redness', 'Dryness']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,8402,P411387,Superfood Antioxidant Cleanser,6169,Youth To The People,404142,4.2112,5851.0,8 oz/ 237 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,2,68.0,14.0,"[Water/Aqua/Eau, Cocamidopropyl Hydroxysultain...","['Vegan', 'Refill Available', 'Good for: Pores...","['Combination', 'Normal', 'Oily']",['Pores']
535,8403,P441644,Mini Superfood Antioxidant Cleanser,6169,Youth To The People,121678,4.2121,5841.0,2 oz/ 59 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,"[Water/Aqua/Eau, Cocamidopropyl Hydroxysultain...","['Refill Available', 'Community Favorite', 'Go...","['Combination', 'Normal', 'Oily']",['Pores']
536,8404,P440307,Superberry Hydrate + Glow Dream Night Mask wit...,6169,Youth To The People,248887,4.3237,2354.0,2 oz/ 59 mL,Size,...,Skincare,Masks,Face Masks,1,18.0,18.0,"[Water/Aqua/Eau, Glycerin, Helianthus Annuus (...","['Vegan', 'Good for: Dullness/Uneven Texture',...",[],['Dullness/Uneven Texture']
537,8405,P461555,Mini Superberry Hydrate + Glow Dream Mask,6169,Youth To The People,79524,4.3237,2354.0,0.5 oz/ 15 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,"[Water/Aqua/Eau, Glycerin, Helianthus Annuus (...","['Good for: Dullness/Uneven Texture', 'allure ...","['Normal', 'Combination', 'Dry']",['Dullness/Uneven Texture']


In [80]:
type(df3['skin_type'].iloc[0])

str

In [81]:
import ast
import pandas as pd

def parse_list(value):
    if pd.isna(value):
        return []
    
    if isinstance(value, list):
        return value
    
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

In [82]:
df3

,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,ingredients_list,highlights_list,skin_type,benefit
0,95,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,6018,Algenist,9314,4.0569,334.0,5 oz/ 150 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[C12-15 Alkyl Benzoate, Ethylhexyl Palmitate, ...",[],[],[]
1,96,P296413,Gentle Rejuvenating Cleanser,6018,Algenist,7681,4.2689,264.0,4 oz/ 120 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Sodium Cocoyl Isethionate, Glyceryl St...",[],[],[]
2,98,P379907,Advanced Anti-Aging Repairing Oil,6018,Algenist,10676,4.4531,245.0,1 oz/ 30 mL,Size,...,Skincare,Moisturizers,Face Oils,0,NaN,NaN,"[Chlorella Protothecoides Oil, Cetearyl Ethylh...",[],[],[]
3,107,P442859,ALIVE Prebiotic Balancing Mask,6018,Algenist,14367,4.3729,118.0,1.7 oz/ 50 mL,Size,...,Skincare,Masks,Face Masks,0,NaN,NaN,"[Glycerin, Water (Aqua, Eau), Sodium Cocoyl Gl...","['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...",['Combination'],['Acne/Blemishes']
4,125,P442546,Balancing Cleanser,6283,Alpha-H,3612,4.5455,77.0,6.25 oz/ 185 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Ethylhexyl Palmitate, Aloe Barbadensis...","['Best for Dry, Combo, Normal Skin', 'Good for...","['Normal', 'Combination', 'Dry']","['Redness', 'Dryness']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,8402,P411387,Superfood Antioxidant Cleanser,6169,Youth To The People,404142,4.2112,5851.0,8 oz/ 237 mL,Size,...,Skincare,Cleansers,Face Wash & Cleansers,2,68.0,14.0,"[Water/Aqua/Eau, Cocamidopropyl Hydroxysultain...","['Vegan', 'Refill Available', 'Good for: Pores...","['Combination', 'Normal', 'Oily']",['Pores']
535,8403,P441644,Mini Superfood Antioxidant Cleanser,6169,Youth To The People,121678,4.2121,5841.0,2 oz/ 59 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,"[Water/Aqua/Eau, Cocamidopropyl Hydroxysultain...","['Refill Available', 'Community Favorite', 'Go...","['Combination', 'Normal', 'Oily']",['Pores']
536,8404,P440307,Superberry Hydrate + Glow Dream Night Mask wit...,6169,Youth To The People,248887,4.3237,2354.0,2 oz/ 59 mL,Size,...,Skincare,Masks,Face Masks,1,18.0,18.0,"[Water/Aqua/Eau, Glycerin, Helianthus Annuus (...","['Vegan', 'Good for: Dullness/Uneven Texture',...",[],['Dullness/Uneven Texture']
537,8405,P461555,Mini Superberry Hydrate + Glow Dream Mask,6169,Youth To The People,79524,4.3237,2354.0,0.5 oz/ 15 mL,Size,...,Skincare,Mini Size,NaN,0,NaN,NaN,"[Water/Aqua/Eau, Glycerin, Helianthus Annuus (...","['Good for: Dullness/Uneven Texture', 'allure ...","['Normal', 'Combination', 'Dry']",['Dullness/Uneven Texture']


In [83]:
import ast
import pandas as pd

def parse_list(value):
    if pd.isna(value):
        return []
    
    if isinstance(value, list):
        return value
    
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

In [84]:
df3['skin_type'] = df3['skin_type'].apply(parse_list)
df3['benefit'] = df3['benefit'].apply(parse_list)

In [85]:
df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 539 entries, 0 to 538
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          539 non-null    int64  
 1   product_id          539 non-null    str    
 2   product_name        539 non-null    str    
 3   brand_id            539 non-null    int64  
 4   brand_name          539 non-null    str    
 5   loves_count         539 non-null    int64  
 6   rating              529 non-null    float64
 7   reviews             529 non-null    float64
 8   size                507 non-null    str    
 9   variation_type      506 non-null    str    
 10  variation_value     494 non-null    str    
 11  variation_desc      6 non-null      str    
 12  ingredients         539 non-null    str    
 13  price_usd           539 non-null    float64
 14  value_price_usd     10 non-null     float64
 15  sale_price_usd      14 non-null     float64
 16  limited_edition    

In [86]:
from collections import Counter

ingredient_counts = Counter()
for lst in df3['ingredients_list']:
    unique_in_product = set(i.lower().strip() for i in lst)
    ingredient_counts.update(unique_in_product)

print('Total unique ingredients:', len(ingredient_counts))

# Keep only ingredients appearing in at least 5 products
MIN_PRODUCTS = 8
frequent_ingredients = {ing for ing, count in ingredient_counts.items() if count >= MIN_PRODUCTS}
print('Ingredients appearing in 8+ products:', len(frequent_ingredients))

Total unique ingredients: 3042
Ingredients appearing in 8+ products: 405


In [87]:
def filter_frequent(lst, frequent_set):
    return [i.lower().strip() for i in lst if i.lower().strip() in frequent_set]

In [88]:
df3['ingredients_filtered'] = df3['ingredients_list'].apply(lambda x: filter_frequent(x, frequent_ingredients))

In [89]:
# Check: did any products lose ALL their ingredients?
print('Products with 0 ingredients after filtering:', (df3['ingredients_filtered'].apply(len) == 0).sum())

Products with 0 ingredients after filtering: 29


In [90]:
empty_after_filter = df3[df3['ingredients_filtered'].apply(len) == 0]
print(len(empty_after_filter))
print(empty_after_filter[['product_name', 'ingredients_list']].head(5))

29
                                          product_name  \
15        Rejuvenating Scalp + Fuller Hair Therapy Set   
16       Cryo Skin Icing Roller + Bright Eyes Gels Set   
89   Snow Mushroom Pore Cleanser with Exfoliating G...   
130          Hyaluronic Marine Hydrating Modeling Mask   
135                                  Cryo Rubber Masks   

                                      ingredients_list  
15                              [Healthy Scalp Serum:]  
16                             [Bright Eyes Eye Gels:]  
89                                              [Bar:]  
130                                          [Step 1:]  
135  [Step 1 Dr.Jart Hyaluronic Acid 1000 ppm Ampou...  


In [91]:
df3 = df3[df3['ingredients_filtered'].apply(len) > 0].copy()
print(df3.shape)  # should be (510, ...)

(510, 33)


In [92]:
df3

,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,ingredients_list,highlights_list,skin_type,benefit,ingredients_filtered
0,95,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,6018,Algenist,9314,4.0569,334.0,5 oz/ 150 mL,Size,...,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[C12-15 Alkyl Benzoate, Ethylhexyl Palmitate, ...",[],[],[],"[c12-15 alkyl benzoate, ethylhexyl palmitate, ..."
1,96,P296413,Gentle Rejuvenating Cleanser,6018,Algenist,7681,4.2689,264.0,4 oz/ 120 mL,Size,...,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Sodium Cocoyl Isethionate, Glyceryl St...",[],[],[],"[water, sodium cocoyl isethionate, glyceryl st..."
2,98,P379907,Advanced Anti-Aging Repairing Oil,6018,Algenist,10676,4.4531,245.0,1 oz/ 30 mL,Size,...,Moisturizers,Face Oils,0,NaN,NaN,"[Chlorella Protothecoides Oil, Cetearyl Ethylh...",[],[],[],"[caprylic/capric triglyceride, retinyl palmita..."
3,107,P442859,ALIVE Prebiotic Balancing Mask,6018,Algenist,14367,4.3729,118.0,1.7 oz/ 50 mL,Size,...,Masks,Face Masks,0,NaN,NaN,"[Glycerin, Water (Aqua, Eau), Sodium Cocoyl Gl...","['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...",[Combination],[Acne/Blemishes],"[glycerin, eau), propanediol, kaolin, bentonit..."
4,125,P442546,Balancing Cleanser,6283,Alpha-H,3612,4.5455,77.0,6.25 oz/ 185 mL,Size,...,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Ethylhexyl Palmitate, Aloe Barbadensis...","['Best for Dry, Combo, Normal Skin', 'Good for...","[Normal, Combination, Dry]","[Redness, Dryness]","[water, ethylhexyl palmitate, sorbitan stearat..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,8402,P411387,Superfood Antioxidant Cleanser,6169,Youth To The People,404142,4.2112,5851.0,8 oz/ 237 mL,Size,...,Cleansers,Face Wash & Cleansers,2,68.0,14.0,"[Water/Aqua/Eau, Cocamidopropyl Hydroxysultain...","['Vegan', 'Refill Available', 'Good for: Pores...","[Combination, Normal, Oily]",[Pores],"[water/aqua/eau, cocamidopropyl hydroxysultain..."
535,8403,P441644,Mini Superfood Antioxidant Cleanser,6169,Youth To The People,121678,4.2121,5841.0,2 oz/ 59 mL,Size,...,Mini Size,NaN,0,NaN,NaN,"[Water/Aqua/Eau, Cocamidopropyl Hydroxysultain...","['Refill Available', 'Community Favorite', 'Go...","[Combination, Normal, Oily]",[Pores],"[water/aqua/eau, cocamidopropyl hydroxysultain..."
536,8404,P440307,Superberry Hydrate + Glow Dream Night Mask wit...,6169,Youth To The People,248887,4.3237,2354.0,2 oz/ 59 mL,Size,...,Masks,Face Masks,1,18.0,18.0,"[Water/Aqua/Eau, Glycerin, Helianthus Annuus (...","['Vegan', 'Good for: Dullness/Uneven Texture',...",[],[Dullness/Uneven Texture],"[water/aqua/eau, glycerin, helianthus annuus (..."
537,8405,P461555,Mini Superberry Hydrate + Glow Dream Mask,6169,Youth To The People,79524,4.3237,2354.0,0.5 oz/ 15 mL,Size,...,Mini Size,NaN,0,NaN,NaN,"[Water/Aqua/Eau, Glycerin, Helianthus Annuus (...","['Good for: Dullness/Uneven Texture', 'allure ...","[Normal, Combination, Dry]",[Dullness/Uneven Texture],"[water/aqua/eau, glycerin, helianthus annuus (..."


In [93]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb_ingredients = MultiLabelBinarizer()
X = mlb_ingredients.fit_transform(df3['ingredients_filtered'])

print('Feature matrix shape:', X.shape)
print('Example feature names:', mlb_ingredients.classes_[:10])

Feature matrix shape: (510, 405)
Example feature names: ['1' '2-hexanediol' 'acacia senegal gum'
 'acer saccharum (sugar maple) extract' 'acetyl glucosamine'
 'acetyl hexapeptide-8' 'acrylates copolymer'
 'acrylates/c10-30 alkyl acrylate crosspolymer' 'adenosine' 'alanine']


In [94]:
df3

,Unnamed: 0,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,...,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,ingredients_list,highlights_list,skin_type,benefit,ingredients_filtered
0,95,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,6018,Algenist,9314,4.0569,334.0,5 oz/ 150 mL,Size,...,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[C12-15 Alkyl Benzoate, Ethylhexyl Palmitate, ...",[],[],[],"[c12-15 alkyl benzoate, ethylhexyl palmitate, ..."
1,96,P296413,Gentle Rejuvenating Cleanser,6018,Algenist,7681,4.2689,264.0,4 oz/ 120 mL,Size,...,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Sodium Cocoyl Isethionate, Glyceryl St...",[],[],[],"[water, sodium cocoyl isethionate, glyceryl st..."
2,98,P379907,Advanced Anti-Aging Repairing Oil,6018,Algenist,10676,4.4531,245.0,1 oz/ 30 mL,Size,...,Moisturizers,Face Oils,0,NaN,NaN,"[Chlorella Protothecoides Oil, Cetearyl Ethylh...",[],[],[],"[caprylic/capric triglyceride, retinyl palmita..."
3,107,P442859,ALIVE Prebiotic Balancing Mask,6018,Algenist,14367,4.3729,118.0,1.7 oz/ 50 mL,Size,...,Masks,Face Masks,0,NaN,NaN,"[Glycerin, Water (Aqua, Eau), Sodium Cocoyl Gl...","['Vegan', 'Hypoallergenic', 'Good for: Acne/Bl...",[Combination],[Acne/Blemishes],"[glycerin, eau), propanediol, kaolin, bentonit..."
4,125,P442546,Balancing Cleanser,6283,Alpha-H,3612,4.5455,77.0,6.25 oz/ 185 mL,Size,...,Cleansers,Face Wash & Cleansers,0,NaN,NaN,"[Water, Ethylhexyl Palmitate, Aloe Barbadensis...","['Best for Dry, Combo, Normal Skin', 'Good for...","[Normal, Combination, Dry]","[Redness, Dryness]","[water, ethylhexyl palmitate, sorbitan stearat..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,8402,P411387,Superfood Antioxidant Cleanser,6169,Youth To The People,404142,4.2112,5851.0,8 oz/ 237 mL,Size,...,Cleansers,Face Wash & Cleansers,2,68.0,14.0,"[Water/Aqua/Eau, Cocamidopropyl Hydroxysultain...","['Vegan', 'Refill Available', 'Good for: Pores...","[Combination, Normal, Oily]",[Pores],"[water/aqua/eau, cocamidopropyl hydroxysultain..."
535,8403,P441644,Mini Superfood Antioxidant Cleanser,6169,Youth To The People,121678,4.2121,5841.0,2 oz/ 59 mL,Size,...,Mini Size,NaN,0,NaN,NaN,"[Water/Aqua/Eau, Cocamidopropyl Hydroxysultain...","['Refill Available', 'Community Favorite', 'Go...","[Combination, Normal, Oily]",[Pores],"[water/aqua/eau, cocamidopropyl hydroxysultain..."
536,8404,P440307,Superberry Hydrate + Glow Dream Night Mask wit...,6169,Youth To The People,248887,4.3237,2354.0,2 oz/ 59 mL,Size,...,Masks,Face Masks,1,18.0,18.0,"[Water/Aqua/Eau, Glycerin, Helianthus Annuus (...","['Vegan', 'Good for: Dullness/Uneven Texture',...",[],[Dullness/Uneven Texture],"[water/aqua/eau, glycerin, helianthus annuus (..."
537,8405,P461555,Mini Superberry Hydrate + Glow Dream Mask,6169,Youth To The People,79524,4.3237,2354.0,0.5 oz/ 15 mL,Size,...,Mini Size,NaN,0,NaN,NaN,"[Water/Aqua/Eau, Glycerin, Helianthus Annuus (...","['Good for: Dullness/Uneven Texture', 'allure ...","[Normal, Combination, Dry]",[Dullness/Uneven Texture],"[water/aqua/eau, glycerin, helianthus annuus (..."


In [95]:
X[0]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,

In [96]:
mlb_skintype = MultiLabelBinarizer()
y_skintype = mlb_skintype.fit_transform(df3['skin_type'])
print('Skin type target shape:', y_skintype.shape)
print('Skin type classes:', mlb_skintype.classes_)

mlb_benefit = MultiLabelBinarizer()
y_benefit = mlb_benefit.fit_transform(df3['benefit'])
print('Benefit target shape:', y_benefit.shape)
print('Benefit classes:', mlb_benefit.classes_)

Skin type target shape: (510, 4)
Skin type classes: ['Combination' 'Dry' 'Normal' 'Oily']
Benefit target shape: (510, 9)
Benefit classes: ['Acne/Blemishes' 'Anti-Aging' 'Dark Circles' 'Dark spots' 'Dryness'
 'Dullness/Uneven Texture' 'Loss of firmness' 'Pores' 'Redness']


In [97]:
y_skintype[0]

array([0, 0, 0, 0])

In [98]:
has_skintype = y_skintype.sum(axis=1) > 0
X_st = X[has_skintype]
y_st = y_skintype[has_skintype]

print('X_st shape:', X_st.shape)
print('y_st shape:', y_st.shape)

X_st shape: (194, 405)
y_st shape: (194, 4)


In [99]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_st, y_st,test_size=0.2, random_state=42)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

Train shape: (155, 405)
Test shape: (39, 405)


In [100]:
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
base_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight='balanced')
model = MultiOutputClassifier(base_model)
model.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [101]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=mlb_skintype.classes_, zero_division=0))

              precision    recall  f1-score   support

 Combination       0.85      0.91      0.88        32
         Dry       0.86      0.75      0.80        24
      Normal       0.85      0.88      0.86        32
        Oily       0.59      0.77      0.67        13

   micro avg       0.81      0.84      0.83       101
   macro avg       0.79      0.83      0.80       101
weighted avg       0.82      0.84      0.83       101
 samples avg       0.80      0.85      0.79       101



In [102]:
print('Rows with at least 1 skin type label:', (df3['skin_type'].apply(len) > 0).sum())

Rows with at least 1 skin type label: 194


In [103]:
print("df3 shape:", df3.shape)
print("df1 shape:", df1.shape)
print()

print("df3['ingredients_list'] type:", type(df3['ingredients_list'].iloc[0]))
print("df3['skin_type'] type:", type(df3['skin_type'].iloc[0]))
print("df3['benefit'] type:", type(df3['benefit'].iloc[0]))
print()

print("X shape:", X.shape)
print("y_skintype shape:", y_skintype.shape)
print("y_benefit shape:", y_benefit.shape)
print()

print("Skin type classes:", mlb_skintype.classes_)
print("Benefit classes:", mlb_benefit.classes_)

df3 shape: (510, 33)
df1 shape: (248, 3)

df3['ingredients_list'] type: <class 'list'>
df3['skin_type'] type: <class 'list'>
df3['benefit'] type: <class 'list'>

X shape: (510, 405)
y_skintype shape: (510, 4)
y_benefit shape: (510, 9)

Skin type classes: ['Combination' 'Dry' 'Normal' 'Oily']
Benefit classes: ['Acne/Blemishes' 'Anti-Aging' 'Dark Circles' 'Dark spots' 'Dryness'
 'Dullness/Uneven Texture' 'Loss of firmness' 'Pores' 'Redness']


In [104]:
# Step 1: Filter to rows with an actual benefit label
has_benefit = y_benefit.sum(axis=1) > 0
X_bn = X[has_benefit]
y_bn = y_benefit[has_benefit]

print('X_bn shape:', X_bn.shape)
print('y_bn shape:', y_bn.shape)

X_bn shape: (357, 405)
y_bn shape: (357, 9)


In [105]:
X_train_bn, X_test_bn, y_train_bn, y_test_bn = train_test_split(
    X_bn, y_bn, test_size=0.2, random_state=42
)

print('Train shape:', X_train_bn.shape)
print('Test shape:', X_test_bn.shape)

Train shape: (285, 405)
Test shape: (72, 405)


In [106]:
base_model_bn = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
model_bn = MultiOutputClassifier(base_model_bn)
model_bn.fit(X_train_bn, y_train_bn)

print("Benefit model trained.")

Benefit model trained.


In [107]:
y_pred_bn = model_bn.predict(X_test_bn)
print(classification_report(y_test_bn, y_pred_bn, target_names=mlb_benefit.classes_, zero_division=0))

                         precision    recall  f1-score   support

         Acne/Blemishes       0.33      0.20      0.25        10
             Anti-Aging       1.00      0.69      0.82        13
           Dark Circles       1.00      0.23      0.38        13
             Dark spots       0.00      0.00      0.00         2
                Dryness       0.68      0.81      0.74        26
Dullness/Uneven Texture       0.70      0.56      0.62        34
       Loss of firmness       1.00      0.13      0.24        15
                  Pores       0.76      0.62      0.68        21
                Redness       0.33      0.33      0.33         3

              micro avg       0.71      0.51      0.60       137
              macro avg       0.65      0.40      0.45       137
           weighted avg       0.75      0.51      0.56       137
            samples avg       0.67      0.54      0.57       137



In [108]:
import numpy as np

# Get predicted probabilities instead of hard yes/no predictions
y_proba_bn = model_bn.predict_proba(X_test_bn)

# predict_proba returns one array per label; extract probability of "yes" (class 1) for each
proba_matrix = np.array([p[:, 1] for p in y_proba_bn]).T
print("Probability matrix shape:", proba_matrix.shape)

# Try a lower threshold (0.3 instead of default 0.5)
threshold = 0.3
y_pred_bn_adjusted = (proba_matrix >= threshold).astype(int)

print(classification_report(y_test_bn, y_pred_bn_adjusted, target_names=mlb_benefit.classes_, zero_division=0))

Probability matrix shape: (72, 9)
                         precision    recall  f1-score   support

         Acne/Blemishes       0.30      0.30      0.30        10
             Anti-Aging       0.82      0.69      0.75        13
           Dark Circles       0.86      0.46      0.60        13
             Dark spots       1.00      0.50      0.67         2
                Dryness       0.56      0.88      0.69        26
Dullness/Uneven Texture       0.57      0.68      0.62        34
       Loss of firmness       0.60      0.20      0.30        15
                  Pores       0.77      0.95      0.85        21
                Redness       0.14      0.33      0.20         3

              micro avg       0.60      0.65      0.62       137
              macro avg       0.62      0.56      0.55       137
           weighted avg       0.63      0.65      0.61       137
            samples avg       0.64      0.67      0.62       137



In [109]:
from sklearn.metrics import f1_score

thresholds = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

print(f"{'Threshold':<12}{'Micro F1':<12}{'Macro F1':<12}{'Weighted F1'}")
for t in thresholds:
    y_pred_t = (proba_matrix >= t).astype(int)
    micro = f1_score(y_test_bn, y_pred_t, average='micro', zero_division=0)
    macro = f1_score(y_test_bn, y_pred_t, average='macro', zero_division=0)
    weighted = f1_score(y_test_bn, y_pred_t, average='weighted', zero_division=0)
    print(f"{t:<12}{micro:<12.3f}{macro:<12.3f}{weighted:.3f}")

Threshold   Micro F1    Macro F1    Weighted F1
0.2         0.596       0.540       0.599
0.25        0.601       0.533       0.595
0.3         0.625       0.553       0.612
0.35        0.627       0.564       0.612
0.4         0.625       0.482       0.600
0.45        0.592       0.456       0.562
0.5         0.596       0.451       0.564


In [110]:
BEST_THRESHOLD_BENEFIT = 0.35

y_pred_final_bn = (proba_matrix >= BEST_THRESHOLD_BENEFIT).astype(int)
print(classification_report(y_test_bn, y_pred_final_bn, target_names=mlb_benefit.classes_, zero_division=0))

                         precision    recall  f1-score   support

         Acne/Blemishes       0.38      0.30      0.33        10
             Anti-Aging       0.90      0.69      0.78        13
           Dark Circles       0.86      0.46      0.60        13
             Dark spots       1.00      0.50      0.67         2
                Dryness       0.56      0.85      0.68        26
Dullness/Uneven Texture       0.63      0.65      0.64        34
       Loss of firmness       0.60      0.20      0.30        15
                  Pores       0.77      0.81      0.79        21
                Redness       0.25      0.33      0.29         3

              micro avg       0.64      0.61      0.63       137
              macro avg       0.66      0.53      0.56       137
           weighted avg       0.66      0.61      0.61       137
            samples avg       0.64      0.63      0.62       137



In [111]:
y_proba_st = model.predict_proba(X_test)
proba_matrix_st = np.array([p[:, 1] for p in y_proba_st]).T

print(f"{'Threshold':<12}{'Micro F1':<12}{'Macro F1':<12}{'Weighted F1'}")
for t in thresholds:
    y_pred_t = (proba_matrix_st >= t).astype(int)
    micro = f1_score(y_test, y_pred_t, average='micro', zero_division=0)
    macro = f1_score(y_test, y_pred_t, average='macro', zero_division=0)
    weighted = f1_score(y_test, y_pred_t, average='weighted', zero_division=0)
    print(f"{t:<12}{micro:<12.3f}{macro:<12.3f}{weighted:.3f}")

Threshold   Micro F1    Macro F1    Weighted F1
0.2         0.829       0.806       0.848
0.25        0.828       0.805       0.842
0.3         0.838       0.816       0.848
0.35        0.844       0.823       0.851
0.4         0.832       0.809       0.838
0.45        0.825       0.802       0.829
0.5         0.825       0.802       0.827


In [112]:
BEST_THRESHOLD_SKINTYPE = 0.35

y_pred_final_st = (proba_matrix_st >= BEST_THRESHOLD_SKINTYPE).astype(int)
print(classification_report(y_test, y_pred_final_st, target_names=mlb_skintype.classes_, zero_division=0))

              precision    recall  f1-score   support

 Combination       0.86      0.94      0.90        32
         Dry       0.83      0.83      0.83        24
      Normal       0.86      0.94      0.90        32
        Oily       0.52      0.92      0.67        13

   micro avg       0.79      0.91      0.84       101
   macro avg       0.77      0.91      0.82       101
weighted avg       0.81      0.91      0.85       101
 samples avg       0.79      0.92      0.83       101



In [113]:
def analyze_ingredients(ingredient_list, df1_lookup):
    """
    Takes a list of ingredient names, checks each against the reference dictionary,
    and returns matched actives/benefits and irritant flags.
    """
    matched_actives = []
    irritant_flags = []
    
    for ing in ingredient_list:
        ing_lower = ing.lower().strip()
        for ref in df1_lookup:
            if ref['name_lower'] in ing_lower:
                if ref['good_for']:
                    matched_actives.append({
                        'ingredient': ref['name'],
                        'good_for': ref['good_for']
                    })
                if ref['avoid']:
                    irritant_flags.append({
                        'ingredient': ref['name'],
                        'avoid_reasons': ref['avoid']
                    })
                break
    
    return {
        'matched_actives': matched_actives,
        'irritant_flags': irritant_flags
    }

In [114]:
sample_ingredients = df3.iloc[0]['ingredients_list']
result = analyze_ingredients(sample_ingredients, df1_lookup)

print("Matched actives:")
for m in result['matched_actives']:
    print(f"  - {m['ingredient']}: {m['good_for']}")

print("\nIrritant flags:")
for f in result['irritant_flags']:
    print(f"  - {f['ingredient']}: {f['avoid_reasons']}")

NameError: name 'df1_lookup' is not defined

In [ ]:
df1_lookup = []
for _, row in df1.iterrows():
    df1_lookup.append({
        'name_lower': row['name'].lower().strip(),
        'name': row['name'],
        'good_for': row['good_for_list'],
        'avoid': row['avoid_list']
    })

print(len(df1_lookup))
print(df1_lookup[0])

AttributeError: 'float' object has no attribute 'lower'

In [ ]:
print('Missing names in df1:', df1['name'].isnull().sum())
print(df1[df1['name'].isnull()])

Missing names in df1: 1
   name good_for_list avoid_list
25  NaN            []         []


In [ ]:
df1

,name,good_for_list,avoid_list
0,Alpha-Glucan Oligosaccharide,"['Acne', 'Blackheads', 'Redness', 'Pregnancy']",['Related Allergy']
1,Aloe Vera,"['Dry and dehydrated skin', 'Impaired skin bar...",['Related Allergy']
2,Allantoin,"['Fine Lines', 'Impaired skin barrier', 'Redne...",['Related Allergy']
3,Algin,"['Dry and dehydrated skin', 'Fine Lines', 'Pre...",['Related Allergy']
4,Algae Extract,"['Dry and dehydrated skin', 'Fine Lines', 'Pre...",['Related Allergy']
...,...,...,...
243,Zinc Sulfate,['Anyone'],['Related Allergy']
244,Zinc PCA,"['Acne', 'Blackheads', 'Enlarged Pores', 'Fine...",['Related Allergy']
245,Neoglucosamine,"['Blackheads', 'Elasticity', 'Enlarged Pores',...","['Related Allergy', 'Impaired skin barrier']"
246,Taurine,"['Dry and dehydrated skin', 'Radiance', 'Pregn...",['Related Allergy']


In [ ]:
print('Missing names in df1:', df1['name'].isnull().sum())
print(df1[df1['name'].isnull()])

Missing names in df1: 1
   name good_for_list avoid_list
25  NaN            []         []


In [ ]:
df1_lookup = []
for _, row in df1.iterrows():
    if pd.isnull(row['name']):
        continue
    df1_lookup.append({
        'name_lower': row['name'].lower().strip(),
        'name': row['name'],
        'good_for': row['good_for_list'],
        'avoid': row['avoid_list']
    })

print(len(df1_lookup))
print(df1_lookup[0])

247
{'name_lower': 'alpha-glucan oligosaccharide', 'name': 'Alpha-Glucan Oligosaccharide', 'good_for': "['Acne', 'Blackheads', 'Redness', 'Pregnancy']", 'avoid': "['Related Allergy']"}


In [ ]:
df1['good_for_list'] = df1['good_for_list'].apply(safe_parse)
df1['avoid_list'] = df1['avoid_list'].apply(safe_parse)

# Confirm it's fixed
print(type(df1['good_for_list'].iloc[0]))

# Now rebuild the lookup with real lists
df1_lookup = []
for _, row in df1.iterrows():
    if pd.isnull(row['name']):
        continue
    df1_lookup.append({
        'name_lower': row['name'].lower().strip(),
        'name': row['name'],
        'good_for': row['good_for_list'],
        'avoid': row['avoid_list']
    })

print(len(df1_lookup))
print(df1_lookup[0])

<class 'list'>
247
{'name_lower': 'alpha-glucan oligosaccharide', 'name': 'Alpha-Glucan Oligosaccharide', 'good_for': ['Acne', 'Blackheads', 'Redness', 'Pregnancy'], 'avoid': ['Related Allergy']}


In [ ]:
def safe_parse(val):
    if isinstance(val, list):
        return val  # already a real list, nothing to do
    if pd.isnull(val):
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

In [ ]:
df1['good_for_list'] = df1['good_for_list'].apply(safe_parse)
df1['avoid_list'] = df1['avoid_list'].apply(safe_parse)

print(type(df1['good_for_list'].iloc[0]))
print(df1['good_for_list'].iloc[0])

<class 'list'>
['Acne', 'Blackheads', 'Redness', 'Pregnancy']


In [ ]:
df1_lookup = []
for _, row in df1.iterrows():
    if pd.isnull(row['name']):
        continue
    df1_lookup.append({
        'name_lower': row['name'].lower().strip(),
        'name': row['name'],
        'good_for': row['good_for_list'],
        'avoid': row['avoid_list']
    })

print(len(df1_lookup))
print(df1_lookup[0])

247
{'name_lower': 'alpha-glucan oligosaccharide', 'name': 'Alpha-Glucan Oligosaccharide', 'good_for': ['Acne', 'Blackheads', 'Redness', 'Pregnancy'], 'avoid': ['Related Allergy']}


In [ ]:
def analyze_ingredients(ingredient_list, df1_lookup):
    matched_actives = []
    irritant_flags = []
    
    for ing in ingredient_list:
        ing_lower = ing.lower().strip()
        for ref in df1_lookup:
            if ref['name_lower'] in ing_lower:
                if ref['good_for']:
                    matched_actives.append({
                        'ingredient': ref['name'],
                        'good_for': ref['good_for']
                    })
                if ref['avoid']:
                    irritant_flags.append({
                        'ingredient': ref['name'],
                        'avoid_reasons': ref['avoid']
                    })
                break
    
    return {
        'matched_actives': matched_actives,
        'irritant_flags': irritant_flags
    }

sample_ingredients = df3.iloc[0]['ingredients_list']
result = analyze_ingredients(sample_ingredients, df1_lookup)

print("Matched actives:")
for m in result['matched_actives']:
    print(f"  - {m['ingredient']}: {m['good_for']}")

print("\nIrritant flags:")
for f in result['irritant_flags']:
    print(f"  - {f['ingredient']}: {f['avoid_reasons']}")

Matched actives:
  - Chlorella Protothecoides Oil: ['Dry and dehydrated skin', 'Fine Lines', 'Pregnancy', 'Wrinkles']
  - Algae Exopolysaccharides: ['Dry and dehydrated skin', 'Fine Lines', 'Pregnancy', 'Wrinkles']
  - Retinyl Palmitate: ['Fine Lines']

Irritant flags:
  - Chlorella Protothecoides Oil: ['Related Allergy']
  - Algae Exopolysaccharides: ['Related Allergy']
  - Retinyl Palmitate: ['Pregnancy', 'Impaired skin barrier']


In [ ]:
import joblib

joblib.dump(model, '../models/skintype_model.pkl')
joblib.dump(model_bn, '../models/benefit_model.pkl')
joblib.dump(mlb_ingredients, '../models/ingredients_encoder.pkl')
joblib.dump(mlb_skintype, '../models/skintype_encoder.pkl')
joblib.dump(mlb_benefit, '../models/benefit_encoder.pkl')

df1.to_csv('../data/processed/ingredients_reference_clean.csv', index=False)

print("All models and encoders saved.")

FileNotFoundError: [Errno 2] No such file or directory: '../models/skintype_model.pkl'

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

import joblib

joblib.dump(model, '../models/skintype_model.pkl')
joblib.dump(model_bn, '../models/benefit_model.pkl')
joblib.dump(mlb_ingredients, '../models/ingredients_encoder.pkl')
joblib.dump(mlb_skintype, '../models/skintype_encoder.pkl')
joblib.dump(mlb_benefit, '../models/benefit_encoder.pkl')

df1.to_csv('../data/processed/ingredients_reference_clean.csv', index=False)

print("All models and encoders saved.")

All models and encoders saved.


In [ ]:
KNOWN_COMMA_INGREDIENTS = ['1,2-Hexanediol', '1,3-Propanediol', '2,3-Butanediol', '1,10-Decanediol']

def smart_split_ingredients(text):
    protected = text
    for ing in KNOWN_COMMA_INGREDIENTS:
        protected = protected.replace(ing, ing.replace(',', '§'))
    
    parts = [p.strip().replace('§', ',') for p in protected.split(',') if p.strip()]
    return parts

In [ ]:
sample = "Water, 1,2-Hexanediol, Niacinamide"
print(smart_split_ingredients(sample))

['Water', '1,2-Hexanediol', 'Niacinamide']


In [ ]:
import joblib
import numpy as np

skintype_model = joblib.load('../models/skintype_model.pkl')
mlb_ingredients = joblib.load('../models/ingredients_encoder.pkl')
mlb_skintype = joblib.load('../models/skintype_encoder.pkl')

print("Loaded successfully")

Loaded successfully


In [ ]:
X_empty = mlb_ingredients.transform([[]])
proba_empty = skintype_model.predict_proba(X_empty)
proba_empty_matrix = np.array([p[:, 1] for p in proba_empty]).T
print("Baseline (no ingredients) probabilities:", dict(zip(mlb_skintype.classes_, proba_empty_matrix[0])))

Baseline (no ingredients) probabilities: {'Combination': np.float64(0.8797469442375091), 'Dry': np.float64(0.6617859566510279), 'Normal': np.float64(0.8951256848215249), 'Oily': np.float64(0.3363050122079799)}


In [115]:
X_empty = mlb_ingredients.transform([[]])
proba_empty = skintype_model.predict_proba(X_empty) if False else model.predict_proba(X_empty)
proba_empty_matrix = np.array([p[:, 1] for p in proba_empty]).T
print("Baseline (no ingredients) probabilities:", dict(zip(mlb_skintype.classes_, proba_empty_matrix[0])))

Baseline (no ingredients) probabilities: {'Combination': np.float64(0.7470342280636546), 'Dry': np.float64(0.6630570885018133), 'Normal': np.float64(0.7902836283837095), 'Oily': np.float64(0.33275323182685546)}


In [116]:
base_model = LogisticRegression(max_iter=1000, C=0.3, random_state=42, class_weight='balanced')
model = MultiOutputClassifier(base_model)
model.fit(X_train, y_train)

X_empty = mlb_ingredients.transform([[]])
proba_empty = model.predict_proba(X_empty)
proba_empty_matrix = np.array([p[:, 1] for p in proba_empty]).T
print("Baseline probabilities:", dict(zip(mlb_skintype.classes_, proba_empty_matrix[0])))

Baseline probabilities: {'Combination': np.float64(0.6694621767191552), 'Dry': np.float64(0.607866551595447), 'Normal': np.float64(0.700556843151423), 'Oily': np.float64(0.3859908459902781)}


In [117]:
y_pred_check = model.predict(X_test)
print(classification_report(y_test, y_pred_check, target_names=mlb_skintype.classes_, zero_division=0))


              precision    recall  f1-score   support

 Combination       0.85      0.91      0.88        32
         Dry       0.82      0.75      0.78        24
      Normal       0.85      0.91      0.88        32
        Oily       0.59      0.77      0.67        13

   micro avg       0.80      0.85      0.83       101
   macro avg       0.78      0.83      0.80       101
weighted avg       0.81      0.85      0.83       101
 samples avg       0.80      0.85      0.80       101



In [118]:
joblib.dump(model, '../models/skintype_model.pkl')
print("Final balanced skin-type model saved.")

Final balanced skin-type model saved.


In [119]:
proba_st = model.predict_proba(X_test)
proba_st_matrix = np.array([p[:, 1] for p in proba_st]).T

print(f"{'Threshold':<12}{'Micro F1':<12}{'Macro F1':<12}{'Weighted F1'}")
for t in [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]:
    y_pred_t = (proba_st_matrix >= t).astype(int)
    micro = f1_score(y_test, y_pred_t, average='micro', zero_division=0)
    macro = f1_score(y_test, y_pred_t, average='macro', zero_division=0)
    weighted = f1_score(y_test, y_pred_t, average='weighted', zero_division=0)
    print(f"{t:<12}{micro:<12.3f}{macro:<12.3f}{weighted:.3f}")

Threshold   Micro F1    Macro F1    Weighted F1
0.4         0.818       0.792       0.831
0.45        0.832       0.810       0.837
0.5         0.827       0.802       0.829
0.55        0.818       0.794       0.817
0.6         0.790       0.770       0.790
0.65        0.732       0.713       0.728
0.7         0.655       0.638       0.648
